In [32]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

import matplotlib.pyplot as plt
import seaborn as sns

In [33]:
temp_df = pd.read_csv('IMDB Dataset.csv')

In [34]:
df=temp_df.iloc[:15000]

In [35]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [36]:
df['review'][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [37]:
df['sentiment'].value_counts()

sentiment
negative    7609
positive    7391
Name: count, dtype: int64

In [38]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [39]:
df.duplicated().sum()

np.int64(39)

In [40]:
df.drop_duplicates(inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_27152\3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [41]:
df.duplicated().sum()

np.int64(0)

In [42]:
#Basic preprocessing
#REMOVE TAGS
#LOWERCASE

In [43]:
def clean_text(text):
    text = re.sub('<.*?>', '', text)   # remove HTML
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    return text

df['review'] = df['review'].apply(clean_text)

C:\Users\HP\AppData\Local\Temp\ipykernel_27152\1024818699.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(clean_text)


In [44]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there s a family where a little boy ...,negative
4,petter mattei s love in the time of money is...,positive
...,...,...
14995,bobcat goldthwait should be commended for atte...,negative
14996,and it s not because since her days on claris...,positive
14997,a traveling couple horton and hamilton stumbl...,negative
14998,this film is deeply disappointing not only th...,negative


In [45]:
X = df['review']
y = df['sentiment']

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [47]:
#BoW
cv = CountVectorizer(max_features=5000)

X_train_bow = cv.fit_transform(X_train)
X_test_bow = cv.transform(X_test)

In [48]:
X_train_bow.shape 

(11968, 5000)

In [49]:
#TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [50]:
#Word2Vec
sentences = []

for doc in X_train:
    sentences.append(simple_preprocess(doc))

In [51]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [52]:
def document_vector(doc):
    words = [word for word in simple_preprocess(doc) if word in w2v_model.wv]

    if len(words) == 0:
        return np.zeros(w2v_model.vector_size)

    return np.mean(w2v_model.wv[words], axis=0)

In [53]:
X_train_w2v = np.array([document_vector(doc) for doc in tqdm(X_train)])
X_test_w2v = np.array([document_vector(doc) for doc in tqdm(X_test)])

100%|█████████████████████████████████████████████████████████████████████████████| 2993/2993 [00:03<00:00, 782.59it/s]


In [54]:
results = {}

def train_and_eval(name, model, X_tr, X_te):
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"\n{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

In [55]:
# BoW
train_and_eval("BoW + LogisticRegression",
               LogisticRegression(max_iter=1000),
               X_train_bow, X_test_bow)

train_and_eval("BoW + NaiveBayes",
               MultinomialNB(),
               X_train_bow, X_test_bow)


# TF-IDF
train_and_eval("TFIDF + LogisticRegression",
               LogisticRegression(max_iter=1000),
               X_train_tfidf, X_test_tfidf)

train_and_eval("TFIDF + NaiveBayes",
               MultinomialNB(),
               X_train_tfidf, X_test_tfidf)


# Word2Vec
train_and_eval("Word2Vec + LogisticRegression",
               LogisticRegression(max_iter=1000),
               X_train_w2v, X_test_w2v)


BoW + LogisticRegression Accuracy: 0.8593
              precision    recall  f1-score   support

    negative       0.86      0.86      0.86      1503
    positive       0.86      0.86      0.86      1490

    accuracy                           0.86      2993
   macro avg       0.86      0.86      0.86      2993
weighted avg       0.86      0.86      0.86      2993


BoW + NaiveBayes Accuracy: 0.8403
              precision    recall  f1-score   support

    negative       0.83      0.85      0.84      1503
    positive       0.85      0.83      0.84      1490

    accuracy                           0.84      2993
   macro avg       0.84      0.84      0.84      2993
weighted avg       0.84      0.84      0.84      2993


TFIDF + LogisticRegression Accuracy: 0.8770
              precision    recall  f1-score   support

    negative       0.88      0.87      0.88      1503
    positive       0.87      0.89      0.88      1490

    accuracy                           0.88      2993
   ma

In [56]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
results_df = results_df.sort_values(by='Accuracy', ascending=False)

results_df


,Model,Accuracy
2,TFIDF + LogisticRegression,0.877046
0,BoW + LogisticRegression,0.859338
3,TFIDF + NaiveBayes,0.857000
1,BoW + NaiveBayes,0.840294
4,Word2Vec + LogisticRegression,0.810224


In [58]:
best_model = LogisticRegression(max_iter=1000)
best_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [65]:
def predict_review(text):
    text = clean_text(text)
    return best_model.predict(tfidf.transform([text]))[0]

# Test
print(predict_review("This movie was amazing"))
print(predict_review("Worst  movie ever"))

positive
negative
